In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import re
import os
from Bio import Entrez
from time import sleep

In [ ]:
def extractInfo(file_name):
    # Updated regular expression to extract components
    pattern = r"(?P<sample>\w+\.\w+)\.raw-raw\.rna\.(?P<assembler>\w+)\.(?P<config>\w+)\.tsv"
    
    # Match the pattern
    match = re.match(pattern, file_name)
    
    # Extract components if the pattern matches
    if match:
        sample = match.group("sample")
        assembler = match.group("assembler")
        config = match.group("config")
        return sample, assembler, config
    return None, None, None

In [ ]:
def curate_df(path_tsv, cois):
    files = glob.glob(path_tsv)
    dfs = []

    for file in files:
        # Check if the file matches any configuration of interest
        if any(conf in file for conf in cois):
            # Read the file into a DataFrame
            df = pd.read_csv(file, sep='\t', names=[
                "query", "target", "seq.Id.", "alnLen", "MM", "gaps", 
                "startQuery", "EndQuery", "startTarget", "EndTarget", 
                "Eval", "bit score", "queryLen", "targetLen", 
                "queryCov", "targetCov"
            ])

            # Extract components from the filename
            name = os.path.basename(file)
            sample, assembler, config = extractInfo(name)

            # Map the assembler to its "nice" version using the mapping dictionary
            assembler_mapped = mapping.get(f"{assembler}.{config}", assembler)

            # Add extracted and mapped information as new columns
            df["sample"] = sample
            df["assembler"] = assembler_mapped
            df["config"] = config

            dfs.append(df)

    big_df = pd.concat(dfs, ignore_index=True)

    return big_df

In [ ]:
def filter_bacteria_identifiers(file_path, df):
    # Read the input file
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Extract identifiers for rows containing "Bacteria"
    identifiers = []
    for line in lines:
        if "Bacteria" in line:
            identifiers.append(line.split()[0])  # Extract the identifier (first element in the line)
    
    # Filter the DataFrame on these identifiers
    filtered_df = df[df['target'].apply(lambda x: any(identifier in x for identifier in identifiers))]

    return filtered_df

In [ ]:
def filter_targets_from_file(df, file_path):
    # Step 1: Get the set of unique targets from the DataFrame
    target_set = set(df['target'])

    # Step 2: Create a set of identifiers from the input file, cleaning up '\x00'
    file_set = set()
    with open(file_path, 'r') as f:
        for line in f:
            identifier = line.split()[0].replace('\x00', '')  # Remove '\x00' from the identifier
            file_set.add(identifier)

    # Step 3: Compute the intersection of the two sets
    matching_ids = target_set.intersection(file_set)

    # Step 4: Filter the DataFrame based on the matching IDs
    filtered_df = df[df['target'].isin(matching_ids)]

    return filtered_df

In [ ]:
# Function to fetch species name from NCBI
def get_species_name(locus):
    try:
        # Search for the locus in the nuccore database
        handle = Entrez.esearch(db="nuccore", term=locus)
        record = Entrez.read(handle)
        handle.close()

        # If no results found, return None
        if not record["IdList"]:
            return (locus, "Not found")

        # Fetch the full record details using the first result ID
        seq_id = record["IdList"][0]
        handle = Entrez.efetch(db="nuccore", id=seq_id, rettype="gb", retmode="text")
        record_text = handle.read()
        handle.close()

        # Extract the species name (it's usually in the "ORGANISM" field)
        lines = record_text.split("\n")
        for i, line in enumerate(lines):
            if line.strip().startswith("ORGANISM"):
                species_name = lines[i].strip().replace("ORGANISM", "").strip()
                return (locus, species_name)

        return (locus, "Species name not found")
    except Exception as e:
        return (locus, f"Error: {e}")

In [ ]:
configsOfInterest = ['megahit.config10', 'carpedeam.configSafe0', 'carpedeam.configUnsafe0', 'penguin.config10', 'spades.config10'] 
# Mapping dictionary for assembler names
mapping = {
    "carpedeam.configUnsafe0": "CarpeDeam\n(unsafe mode)",
    "carpedeam.configSafe0": "CarpeDeam\n(safe mode)",
    "penguin.config10": "PenguiN",
    "megahit.config10": "MEGAHIT",
    "spades.config10": "metaSPAdes",
}

In [ ]:
path="/data/empirical/results/assembly-rna/silva/*tsv"

In [ ]:
revi = curate_df(path, configsOfInterest)

In [ ]:
# Example usage
bacteria_ids_file = '/rnadb/mmseqsSilva/bacteria_ids.txt'  # The file created by the bash command
filtered_df = filter_targets_from_file(revi, bacteria_ids_file)

In [ ]:
Entrez.email = "email@server.suffix"

In [ ]:
df4 = filtered_df.copy()
df4 = df4[(df4["targetCov"] >= 0.99) & (df4["seq.Id."] >= 0.95) & (df4["sample"] == "EMN001.A0101")]

df4.sort_values(by="seq.Id.", ascending=False, inplace=True)
df4.drop_duplicates(subset="query", inplace=True)
loci_list_4 = df4["target"].str.split(".").str[0].tolist()
unique_loci_4 = list(set(loci_list_4))

In [ ]:
unique_loci_4

In [ ]:
# Retrieve species names for all loci
results = {}
for locus in unique_loci_4:
    results[locus] = get_species_name(locus)
    sleep(1)

### Create dictionary { genus1 : [loci1, loci2, ...], genus2 : [...]}

In [ ]:
# Create the new dictionary
genus_dict = {}

for key, (locus, species) in results.items():
    genus = species  # Extract the first word (genus)
    
    if genus not in genus_dict:
        genus_dict[genus] = []
    
    genus_dict[genus].append(locus)

In [ ]:
#genus_dict

### Create dictionary { query1 : [genus1, genus2, ...], query2 : [...]}

In [ ]:
# Create a lookup: map each locus to its genus
locus_to_genus = {locus: genus for genus, loci in genus_dict.items() for locus in loci}

# Define unwanted genus names
excluded_genera = {"uncultured", "metagenome", "Not", "Unknown.", "unidentified", "bacterium"}

# Dictionary to store results: mapping each query to a set of (genus, seq_id, targetCov) triples.
query_to_genera = {}

for _, row in df4.iterrows():
    query = row["query"]
    target_locus = row["target"].split(".")[0]  # Extract locus (before the first dot)
    seq_id = row["seq.Id."]      # This is a float
    targetCov = row["targetCov"] # This is a float
    
    # Lookup genus; default to "Unknown" if not found
    genus = locus_to_genus.get(target_locus, "Unknown")
    
    triple = (genus, seq_id, targetCov)
    query_to_genera.setdefault(query, set()).add(triple)

# Remove unwanted genera from each query's set
for query in query_to_genera:
    query_to_genera[query] = {triple for triple in query_to_genera[query] if triple[0] not in excluded_genera}

# For each query, for each genus present, keep only the triple with the highest seq_id.
final_query_dict = {}

for query, triples in query_to_genera.items():
    # Temporary dict to hold the best triple per genus
    genus_to_best = {}
    for genus, seq_id, targetCov in triples:
        # seq_id is a float so we can directly compare it
        if genus not in genus_to_best or seq_id > genus_to_best[genus][1]:
            genus_to_best[genus] = (genus, seq_id, targetCov)
    final_query_dict[query] = set(genus_to_best.values())

# Print out queries that map to multiple genera (unexpected cases)
print("Queries mapping to multiple genera (after filtering and selecting highest seq.Id.):")
for query, triples in final_query_dict.items():
    if len(triples) > 1:
        print(f"{query}: {triples}")

In [ ]:
# Read the counts file (adjust filename and separator as needed)
counts_df = pd.read_csv("/data/rRNA/yates_emn_genera.sd02.csv", sep="\t")
# Create a dictionary mapping taxon names to counts (assuming second column header is 'EMN001.A0101')
counts_dict = dict(zip(counts_df['Taxon'], counts_df['EMN001.A0101']))

# Path where Decontam files are located
decontam_path = "/data/rRNA"

# Find all tsv files that match "Decontam*.tsv"
decontam_files = glob.glob(os.path.join(decontam_path, "decontam*.tsv"))

# Build a set of contaminant genera (all in lowercase)
contaminant_genera = set()
for file in decontam_files:
    df = pd.read_csv(file, sep="\t")
    # Assume that the column containing genus names is called "value"
    # Drop NaN values and add each unique genus (lowercase) to the set
    for genus in df['value'].dropna().unique():
        contaminant_genera.add(genus.lower())

# Create the final filtered dictionary
filtered_dict = {}

for acc, (accession, species_name) in results.items():
    # Extract the genus (assumed to be the first word) and convert to lowercase
    genus = species_name.split()[0].lower()
    
    found_with_nonzero = False
    for taxon, count in counts_dict.items():
        if genus in taxon.lower():
            if int(count) > 0:
                found_with_nonzero = True
                break
    if found_with_nonzero:
        continue
    
    # 3. Check if the genus is in the contaminant genera set
    if genus in contaminant_genera:
        continue
    
    # If none of the conditions above matched, keep this entry.
    filtered_dict[acc] = (accession, species_name)

In [ ]:
# Filter counts_df to only include taxa with nonzero counts
nonzero_counts_df = counts_df[counts_df['EMN001.A0101'].astype(int) > 0]

# Extract genera from the counts file (all lower case)
# Assumes the 'Taxon' column holds the genus names
counts_genera = set(nonzero_counts_df['Taxon'].str.lower())

# Extract genera from the results dictionary (assumes each species_name has the genus as the first word)
results_genera = set()
for acc, (accession, species_name) in results.items():
    genus = species_name.split()[0].lower()
    results_genera.add(genus)

# Find genera that are in the counts file (with nonzero counts) but not in the results dictionary
missing_genera = counts_genera - results_genera

In [ ]:
new_taxa = list(filtered_dict.values())

In [ ]:
genera = set([x[1].split(" ")[0] for x in new_taxa])

In [ ]:
final_query_dict